# 01 — HIVDB surveillance data coverage

This notebook inspects a reproducible **2015 published surveillance snapshot** associated with Stanford HIVDB. It focuses on provenance, study coverage, and what can—and cannot—be inferred at country level.

**Epidemiologic caution:** a country on this map means published surveillance evidence is represented in the source. It does not imply nationally representative sampling.

In [ ]:
from pathlib import Path
import json
import pandas as pd

DATA = Path("data")
summary_path = DATA / "hivdb_country_summary_2015.csv"
studies_path = DATA / "hivdb_surveillance_studies_2015.csv"
study_countries_path = DATA / "hivdb_study_countries_2015.csv"
metadata_path = DATA / "hivdb_2015_metadata.json"

required = [summary_path, studies_path, study_countries_path, metadata_path]
missing = [p.name for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Generated data are missing: " + ", ".join(missing) +
        ". In the deployed GitHub Pages site these files are generated during the build. "
        "For local use, run: python scripts/fetch_plos_2015.py --output-dir content/data"
    )

summary = pd.read_csv(summary_path)
studies = pd.read_csv(studies_path)
study_countries = pd.read_csv(study_countries_path)
metadata = json.loads(metadata_path.read_text())


In [ ]:
print("Source DOI:", metadata["doi"])
print("Retrieved:", metadata["retrieved_utc"])
print("Studies:", metadata["n_studies"])
print("Parsed ISO-3 countries:", metadata["n_iso3_countries"])
print("Countries with weighted estimates:", metadata["n_weighted_countries"])
print("Unmatched labels:", metadata["unmatched_country_labels"][:20])

## Country evidence coverage

`n_studies_total` counts any study associated with the country. `n_studies_single_country` is the subset whose denominator can be assigned to that country without guessing.

In [ ]:
coverage_cols = ["country", "iso3", "n_studies_total", "n_studies_single_country", "has_weighted_estimate"]
summary[coverage_cols].sort_values(["n_studies_total", "country"], ascending=[False, True]).head(30)

In [ ]:
print("Countries represented:", summary["iso3"].nunique())
print("Countries with at least one single-country study:", int(summary["has_weighted_estimate"].sum()))
print("Countries represented only through multi-country studies:", int((~summary["has_weighted_estimate"]).sum()))

## Study sample size and calendar time

These distributions help identify where a visually prominent map estimate may be supported by only a few or older studies.

In [ ]:
import matplotlib.pyplot as plt

studies["participants"].dropna().plot.hist(bins=30, title="Study sample-size distribution")
plt.xlabel("Participants")
plt.show()

studies["median_sample_year"].dropna().plot.hist(bins=20, title="Median sample year distribution")
plt.xlabel("Median sample year")
plt.show()

## Multi-country rows

The rows below are retained for provenance and coverage but excluded from weighted country prevalence unless the source provides country-specific denominators.

In [ ]:
multi = study_countries.loc[~study_countries["is_single_country"].astype(bool), [
    "study_id", "reference", "countries", "country_source_raw", "participants", "tdr_overall_pct"
]]
multi.head(25)

### Suggested teaching discussion

Ask: *What does the map unit represent? What denominator is available? How old are the studies? Is publication density the same as disease burden?* These questions are often more important than the mapping library itself.